In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
data = '../data/prices.csv'

In [3]:
df = pd.read_csv(data)

In [4]:
df.head().T

,0,1,2,3,4
date,2008-01-15,2008-01-15,2008-01-15,2008-01-15,2008-01-15
admin1,Kachin,Kachin,Kachin,Kachin,Kachin
admin2,Myitkyina,Myitkyina,Myitkyina,Myitkyina,Myitkyina
market,Wai Maw,Wai Maw,Wai Maw,Wai Maw,Wai Maw
market_id,1021,1021,1021,1021,1021
latitude,25.34,25.34,25.34,25.34,25.34
longitude,97.44,97.44,97.44,97.44,97.44
category,cereals and tubers,"meat, fish and eggs","meat, fish and eggs",miscellaneous food,vegetables and fruits
commodity,Rice (low quality),Meat (chicken),Meat (pork),Salt,Onions
commodity_id,145,94,140,185,173


In [5]:
print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 61790 entries, 0 to 61789
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          61790 non-null  str    
 1   admin1        61790 non-null  str    
 2   admin2        61790 non-null  str    
 3   market        61790 non-null  str    
 4   market_id     61790 non-null  int64  
 5   latitude      61790 non-null  float64
 6   longitude     61790 non-null  float64
 7   category      61790 non-null  str    
 8   commodity     61790 non-null  str    
 9   commodity_id  61790 non-null  int64  
 10  unit          61790 non-null  str    
 11  priceflag     61790 non-null  str    
 12  pricetype     61790 non-null  str    
 13  currency      61790 non-null  str    
 14  price         61790 non-null  float64
 15  usdprice      61790 non-null  float64
dtypes: float64(4), int64(2), str(10)
memory usage: 7.5 MB
None


In [6]:
df['date'] = pd.to_datetime(df['date'])

## Turn Everything into small letters

In [7]:
df.columns = df.columns.str.lower().str.replace(' ', '_')


for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].str.lower().str.replace(' ', '_')


In [36]:
unique_commodities = df['commodity'].unique()
print(f"Total unique commodities: {unique_commodities}")

commodity_dfs = {}

for comm in unique_commodities:
    commodity_dfs[comm] = df[df['commodity'] == comm].copy()

unique_markets = df['market'].unique()
print(f"Total unique market: {unique_markets}")

Total unique commodities: <StringArray>
[                           'rice_(low_quality)',
                                'meat_(chicken)',
                                   'meat_(pork)',
                                          'salt',
                                        'onions',
                                      'tomatoes',
                                      'potatoes',
                                    'oil_(palm)',
                             'chickpeas_(local)',
                                         'maize',
                           'rice_(high_quality)',
                                      'soybeans',
                                        'garlic',
                                   'meat_(beef)',
                               'oil_(groundnut)',
                         'oil_(mixed,_imported)',
                                  'rice_(emata)',
                                     'chickpeas',
                                        'pulses',
          

In [11]:
# Create a dictionary where keys are (commodity, market) tuples
comm_market_dfs = {}

for comm in unique_commodities:
    for mkt in unique_markets:
        # Filter for the specific commodity and market
        subset = df[(df['commodity'] == comm) & (df['market'] == mkt)]
        
        # Only save it if data actually exists for this combination
        if not subset.empty:
            comm_market_dfs[(comm, mkt)] = subset.copy()



In [29]:
commodity_market_dates = (
    df.groupby(['commodity', 'market'])['date']
      .agg(min_date='min', max_date='max')
      .reset_index()
      .sort_values(['commodity', 'market'])
)

display(commodity_market_dates) 

,commodity,market,min_date,max_date
0,chickpeas,yangon,2011-01-15,2020-02-15
1,chickpeas_(local),ah_nauk_pyin,2020-11-15,2020-11-15
2,chickpeas_(local),ah_nauk_ywe,2020-11-15,2020-11-15
3,chickpeas_(local),ahpauk_wa,2020-11-15,2020-11-15
4,chickpeas_(local),alel_than_kyaw,2008-01-15,2011-01-15
...,...,...,...,...
2500,"wage_(non-qualified_labour,_non-agricultural)",wai_maw,2014-01-15,2020-03-15
2501,"wage_(non-qualified_labour,_non-agricultural)",wang_leng,2014-01-15,2017-04-15
2502,"wage_(non-qualified_labour,_non-agricultural)",yaesagyo,2014-01-15,2017-12-15
2503,"wage_(non-qualified_labour,_non-agricultural)",yenanchaung,2014-01-15,2016-08-15


In [ ]:
commodity_market_dates_pivot = commodity_market_dates.pivot(
    index='commodity',
    columns='market',
    values=['min_date', 'max_date']
)



In [31]:
max_dates = df.groupby(['commodity', 'market'])['date'].max().rename('max_date')

pairs_max_date_2026 = (
    max_dates[max_dates.dt.year == 2026]
    .reset_index()
    .sort_values(['commodity', 'market'])
)
    
display(pairs_max_date_2026)

,commodity,market,max_date
0,eggs_(local),aungpan_myoma,2026-09-15
1,eggs_(local),baw_du_pha,2026-06-15
2,eggs_(local),bhamo,2026-09-15
3,eggs_(local),bhamo_market,2026-08-15
4,eggs_(local),bi_kin,2026-04-15
...,...,...,...
867,tomatoes_(local),ywarma,2026-09-15
868,tomatoes_(local),zami_yadana,2026-09-15
869,tomatoes_(local),zamyinzwel,2026-09-15
870,tomatoes_(local),zay_cho,2026-06-15


In [39]:
max_dates.index.get_level_values('commodity').nunique()

25

In [40]:
coverage = (
    df.groupby(['commodity', 'market'])
    .agg(
        first_date=('date', 'min'),
        last_date=('date', 'max'),
        observations=('date', 'count'),
        unique_months=('date', lambda x: x.dt.to_period('M').nunique())
    )
    .reset_index()
)

coverage = coverage.sort_values(
    'unique_months',
    ascending=False
)

display(coverage.head(20))

,commodity,market,first_date,last_date,observations,unique_months
1694,rice_(low_quality),lashio_market,2008-01-15,2026-09-15,202,202
1990,salt,lashio_market,2008-04-15,2026-09-15,179,179
690,oil_(palm),buthidaung,2008-01-15,2026-09-15,168,168
1616,rice_(low_quality),buthidaung,2008-01-15,2026-08-15,162,162
1788,rice_(low_quality),pakokku_myo_ma_market,2014-01-15,2026-09-15,146,146
2082,salt,pakokku_myo_ma_market,2014-01-15,2026-09-15,146,146
1634,rice_(low_quality),hakha_myoma_market,2010-06-15,2025-03-15,139,139
1930,salt,hakha_myoma_market,2010-06-15,2025-02-15,139,139
1912,salt,buthidaung,2011-02-15,2026-09-15,138,138
1708,rice_(low_quality),magway,2008-01-15,2020-03-15,135,135


In [41]:
coverage['unique_months'].describe()

count    2505.000000
mean       24.666667
std        25.663719
min         1.000000
25%         5.000000
50%        14.000000
75%        40.000000
max       202.000000
Name: unique_months, dtype: float64